In [28]:
import numpy as np
import qutip as qt
import sys
import shutil
import os

# parameter class
from Class_Parameters.class_parameters import Parameters

# logging class
from Logging.logging import Logging

# time grid factory
from factory_timegrid import factory_timegrid

# circuit factory
from Circuits.factory_circuits import factory_circuits

from helpers_classical import plot_N_in_1
from helpers_classical import write_parameters_run
# config-file (as arrays, if we want to switch between ini-files quickly)
config_filenames = ['configfile_VCO.ini']

#relative path of root folder
output_path = '../Output/'

# create a parameter object
param_obj = Parameters()

# read input file and set up all parameters
param_obj.setup_all_parameters(config_filenames[0])

# time evolution
from TimeEvolutionClassical.propagator_classical import PropagatorClassical


# create a logging object
# which creates a root folder to store results
logging = Logging(param_obj, output_path=output_path)

shutil.copyfile(os.getcwd() +"/"+config_filenames[0], logging.root + "/configfile.txt")

logging here: we have deleted all files in root folder (but not in its sub-folders)
              we will overwrite subfolders if same name exists


'/home/ldanner/AA_PostDoc/Code/python_code/../Output/VCO/debug/test/configfile.txt'

In [29]:
for N_now in range(param_obj.N_runs):

    # setup new (full and flattened dictionary in parameter object)
    param_obj.setup_current_parameters(N_now)

    print(param_obj.params_now_flat["circuit_name"])
    
    # call the logging object, which will create a new sub-folder
    logging.notify_new_run(N_now, circuit_name=param_obj.params_now_flat["circuit_name"])

    # store the exact run parameters for this run
    write_parameters_run(param_obj.params_now, logging)

    # which parameter is varied + value of it
    vary_name = param_obj.params_now_flat["folder_vary"]
    vary_val = param_obj.params_now_flat[vary_name]

    # --- time grid
    times = factory_timegrid(**param_obj.params_now_flat)
    
    # steady state times
    t_st = param_obj.params_now_flat["t_st"]
    i_st = np.where(times >= t_st)[0][0]  

    # zoom times
    t_z0 = param_obj.params_now_flat["t_z0"]
    t_zf = param_obj.params_now_flat["t_zf"]
    i_z0 = np.where(times >= t_z0)[0][0]  
    i_zf = np.where(times >= t_zf)[0][0]  

    # --- circuit
    circuit = factory_circuits(**param_obj.params_now_flat)

    # --- propagator
    propagator = PropagatorClassical(times, circuit, nsteps=1000,
                                     **param_obj.params_now_flat)

    # propagate
    if param_obj.params_now_flat["load_sol"]:

        propagator.res = np.load(logging.subdir + "\\res.npy")

    else:

        propagator.propagate()

        if param_obj.params_now_flat["store_sol"]:

            np.save(logging.subdir + "/res", propagator.res)

 
    # plots in time space
    plot_N_in_1(times, propagator.res,
                xlabel='time ' + r"$ t \omega_0$", ylabel='',
                fname=logging.subdir + "/res_t.pdf",
                vline=[t_ss])
    

VCO
build rhs of system


AttributeError: 'Circuit_VCO' object has no attribute 'version'